<a href="https://colab.research.google.com/github/noobmaster-ru/diploma/blob/main/Computational_Methods_in_Mathematical_Economics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CMME - Computational Methods in Mathematical Economics

## Наша исходная система


\begin{cases}
\LARGE
\LARGE\frac{C_{t+1}}{C_{t}} = β(1+R_{t+1} - δ) \\
\LARGE K_{t+1} = I_{t} + (1 - δ)K_t \\
\LARGE Y_t = w_tL_t + R_tK_t = AK_t^{\alpha}L_t^{(1-\alpha)} \\
\LARGE L_{t} = (\frac{w_t}{𝛗C_t})^{\frac{1}{\psi}} \\
\LARGE R_t = α\frac{Y_t}{K_t} \\
\LARGE w_t = (1-α)\frac{Y_t}{L_t} \\
\LARGE F_t = AK_t^{\alpha}L_t^{1 - \alpha} = Y_t \\
\LARGE Y_t = C_t + S_t \quad,\quad S_t = I_t, \quad \LARGE S_t = Y_t - C_t \\
\end{cases}

\begin{cases}
\LARGE F_{t} = A*K_{t}^{\alpha}*L_{t}^{1-α} \\
\LARGE K_{t+1} = F(K_{t},L_{t}) + (1-δ)K_{t} - C_{t} \\
\LARGE \frac{C_{t+1}}{C_{t}} = β (\frac{\partial F(K_{t+1},L_{t+1})}{\partial K_{t+1}}  + 1 - δ) \\
\LARGE L_{t} = (\frac{W_t}{𝛗C_t})^{\frac{1}{\psi}} \\
\LARGE F(K_{t},L_{t}) = C_{t} + I_{t}
\end{cases}

## Импорты

In [101]:
import numpy as np
import math
from matplotlib import pyplot as plt
from dataclasses import dataclass
import plotly.graph_objects as go
from scipy.optimize import minimize_scalar, fsolve, root_scalar
from plotly.subplots import make_subplots


EPS = 1e-12
INF = 1e12

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

In [78]:
!pip uninstall -y kaleido
!pip install kaleido==0.2.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 8.9 MB/s eta 0:00:00


## Params

In [79]:
@dataclass
class Params:
    # ==================== LEGACY ====================
    beta: float = 0.96 # discount factor
    alpha: float = 0.33 # elasticity of production wrt capital
    A: float = 0.95 # TFP
    sigma: float = 1.0 # 1/elasticity of intertemporal substitution
    delta: float = 0.1 # depreciation rate
    # ==================== LEGACY ====================
    phi: float = 0.75 # φ - фактор ненависти к труду
    psi: float = 0.35 # ψ - величина, обратная к эластичности труда по Фришу


## steady_state()

In [80]:
def steady_state(p: Params) -> dict:
    R_star = 1.0 / p.beta - 1.0 + p.delta
    v = R_star / p.alpha - p.delta
    L_star = (((1.0 - p.alpha) * R_star) / (p.alpha * p.phi * v)) ** (1.0 / (p.psi + 1.0))
    K_star = ((p.alpha * p.A * L_star**(1.0 - p.alpha)) / R_star) ** (1.0 / (1.0 - p.alpha))
    C_star = v * K_star
    Y_star = p.A * K_star**p.alpha * L_star**(1.0 - p.alpha)
    I_star = p.delta * K_star
    w_star = (1.0 - p.alpha) * Y_star / L_star
    return {
        "R_star": R_star, "v": v, "K_star": K_star, "C_star": C_star,
        "L_star": L_star, "Y_star": Y_star, "I_star": I_star, "S_star": I_star, "w_star": w_star,
    }

## Функции-расчёты

###$\LARGE L_t(K_t,C_t)=
\left(
\frac{(1-\alpha)A}{\phi}\frac{K_t^\alpha}{C_t}
\right)^{\frac{1}{\psi+\alpha}}$

In [81]:
def L_from_KC(K, C, p: Params):
    if K <= 0 or C <= 0:
        return np.nan
    base = ((1.0 - p.alpha) * p.A * K**p.alpha) / (p.phi * C)
    if base <= 0 or not np.isfinite(base):
        return np.nan
    return base ** (1.0 / (p.psi + p.alpha))


###$\LARGE Y_t(K_t,C_t) = A*K_t^{\alpha}*L_t^{(1 - \alpha)} = A*K_t^{\alpha}*L_t(K_t,C_t)$

In [82]:
def Y_from_KC(K, C, p: Params):
    L = L_from_KC(K, C, p)
    if np.isnan(L):
        return np.nan
    return p.A * K**p.alpha * L**(1.0 - p.alpha)

###$\LARGE R_t(K_t,C_t) = \alpha*\frac{Y_t}{K_t}$

In [83]:
def R_from_KC(K, C, p: Params):
    Y = Y_from_KC(K, C, p)
    return p.alpha * Y / K

###$\LARGE w_t(K_t,C_t) = (1-\alpha)*\frac{Y_t}{L_t}$

In [84]:
def w_from_KC(K, C, p: Params):
    L = L_from_KC(K, C, p)
    Y = Y_from_KC(K, C, p)
    if np.isnan(L) or np.isnan(Y):
        return np.nan
    return (1.0 - p.alpha) * Y / L

###$\LARGE I_t(Y_t,C_t) = Y_t - C_t$

In [85]:
def I_from_KC(K, C, p: Params):
    Y = Y_from_KC(K, C, p)
    if np.isnan(Y):
        return np.nan
    return Y - C

###$\LARGE K_{t+1}(I_t,K_t) = I_t + (1 - \delta)K_t$

In [86]:
def K_next_from_KC(K, C, p: Params):
    I = I_from_KC(K, C, p)
    if np.isnan(I):
        return np.nan
    return I + (1.0 - p.delta) * K

##Метод Ньютона


###$\LARGE f(c) = \frac{c}{C_t} - β(1 + R_{t+1}(K_{t+1},c) - \delta) = 0$

In [87]:
def euler_residual(C_next, K_next, C_t, p: Params):
    if C_next <= 0 or K_next <= 0 or C_t <= 0:
        return np.nan
    R_next = R_from_KC(K_next, C_next, p)
    if not np.isfinite(R_next):
        return np.nan
    return C_next / C_t - p.beta * (1.0 + R_next - p.delta)

###$\LARGE f'(c)=\frac{1}{C_t}+\beta \frac{1-\alpha}{\psi+\alpha}\frac{R(K_{t+1},c)}{c}$

(вывод прописан в тексте ВКР)

In [88]:
def euler_residual_prime(C_next, K_next, C_t, p: Params):
    if C_next <= 0 or K_next <= 0 or C_t <= 0:
        return np.nan
    R_next = R_from_KC(K_next, C_next, p)
    if not np.isfinite(R_next):
        return np.nan
    # Производная по C_next (упрощённо)
    return (1.0 / C_t) + p.beta * ((1.0 - p.alpha) / (p.psi + p.alpha)) * (R_next / C_next)


###$\LARGE c^{(n+1)}=c^{(n)}- \lambda_n \frac{f(c^{(n)})}{f'(c^{(n)})}, \quad \lambda_n \in [0,1]$

для

$\LARGE \frac{C_{t+1}}{C_t}=\beta\left(1+R(K_{t+1},C_{t+1})-\delta\right), \quad K_{t+1} = const, \quad C_t = const$

In [89]:
def solve_C_next_newton_damped(
    K_next, C_t, p: Params, tol=1e-4, max_iter=200, lambda_init=1.0, lambda_min=1e-6, verbose=False
):
    if K_next <= 0 or C_t <= 0:
        return np.nan
    c = C_t
    for it in range(max_iter):
        f_val = euler_residual(c, K_next, C_t, p)
        fp_val = euler_residual_prime(c, K_next, C_t, p)
        if not np.isfinite(f_val) or not np.isfinite(fp_val):
            return np.nan
        if abs(f_val) < tol:
            return c
        if abs(fp_val) < 1e-14:
            return np.nan
        step = f_val / fp_val
        lam = lambda_init
        accepted = False
        while lam >= lambda_min:
            c_new = c - lam * step
            if np.isfinite(c_new) and c_new > 0:
                f_new = euler_residual(c_new, K_next, C_t, p)
                if np.isfinite(f_new) and abs(f_new) < abs(f_val):
                    accepted = True
                    break
            lam *= 0.5
        if not accepted:
            return np.nan
        if abs(c_new - c) < tol:
            return c_new
        c = c_new
    return np.nan

In [90]:
def solve_C_next(K_next, C_t, p: Params, tol=1e-8):
    from scipy.optimize import root_scalar
    def res(x):
        return euler_residual(x, K_next, C_t, p)
    try:
        sol = root_scalar(res, bracket=[1e-6, 1e6], method='brentq', xtol=tol)
        if sol.converged:
            return sol.root
        else:
            return np.nan
    except:
        return np.nan

In [91]:
def solve_C_next_fixed_point(K_next, C_t, p: Params, max_iter=50, tol=1e-10):
    c = C_t
    for _ in range(max_iter):
        R_next = R_from_KC(K_next, c)
        if not np.isfinite(R_next):
            return np.nan
        c_new = C_t * p.beta * (1.0 + R_next - p.delta)
        if c_new <= 0 or not np.isfinite(c_new):
            return np.nan
        if abs(c_new - c) < tol:
            return c_new
        c = c_new
    return c

## =======================================================


In [99]:
def simulate_and_check(K0, C0, p, max_periods=2000, tol=1e-4):
    ss = steady_state(p)
    K_ss, C_ss, L_ss = ss['K_star'], ss['C_star'], ss['L_star']

    K = [float(K0)]
    C = [float(C0)]
    L = [L_from_KC(K0, C0, p)]
    Y = [Y_from_KC(K0, C0, p)]                     # <-- добавили
    if np.isnan(L[0]) or np.isnan(Y[0]):
        return False, None, None, None, None, 'diverged'

    for t in range(1, max_periods+1):
        K_next = K_next_from_KC(K[-1], C[-1], p)
        if np.isnan(K_next) or K_next <= 0:
            return False, None, None, None, None, 'diverged'

        C_next = solve_C_next(K_next, C[-1], p)
        if np.isnan(C_next) or C_next <= 0:
            return False, None, None, None, None, 'diverged'

        L_next = L_from_KC(K_next, C_next, p)
        Y_next = Y_from_KC(K_next, C_next, p)     # <-- добавили
        if np.isnan(L_next) or np.isnan(Y_next):
            return False, None, None, None, None, 'diverged'

        K.append(K_next)
        C.append(C_next)
        L.append(L_next)
        Y.append(Y_next)

        # Проверка уменьшения (как у научника)
        if K[-1] - K[-2] < 0:
            return False, np.array(K), np.array(C), np.array(L), np.array(Y), 'K_decrease'
        if C[-1] - C[-2] < 0:
            return False, np.array(K), np.array(C), np.array(L), np.array(Y), 'C_decrease'

        # Сходимость к SS
        if (abs(K[-1] - K_ss) < tol and
            abs(C[-1] - C_ss) < tol and
            abs(L[-1] - L_ss) < tol):
            return True, np.array(K), np.array(C), np.array(L), np.array(Y), 'converged'

    return False, np.array(K), np.array(C), np.array(L), np.array(Y), 'max_periods'

def find_C0_by_shooting(K0, p, c_min=None, c_max=None, max_iter=100, tol=1e-4):
    ss = steady_state(p)
    if c_min is None:
        c_min = 1e-6
    if c_max is None:
        c_max = ss['C_star'] * 2.0

    for it in range(max_iter):
        C0 = (c_min + c_max) / 2.0
        success, K_arr, C_arr, L_arr, Y_arr, reason = simulate_and_check(K0, C0, p, tol=tol)

        if success:
            print(f"Найдено C0 = {C0:.6f} за {it+1} итераций")
            T = len(K_arr) - 1   # количество переходов
            path = {'K': K_arr, 'C': C_arr, 'L': L_arr, 'Y': Y_arr}
            return C0, path, T

        # Корректировка интервала как у научника
        if reason == 'K_decrease':
            c_max = C0
        elif reason == 'C_decrease':
            c_min = C0
        else:
            c_max = C0

        if c_max - c_min < 1e-14:
            break

    print("Не удалось подобрать C0")
    return np.nan, None, None

def plot_trajectories(K_arr, C_arr, L_arr, p):
    """Построение 2D и 3D графиков."""
    ss = steady_state(p)
    t = np.arange(len(K_arr))

    plt.figure(figsize=(12,8))
    plt.subplot(3,1,1)
    plt.plot(t, K_arr, label='K')
    plt.axhline(ss['K_star'], color='r', ls='--', label='SS K')
    plt.legend()
    plt.subplot(3,1,2)
    plt.plot(t, C_arr, label='C')
    plt.axhline(ss['C_star'], color='r', ls='--', label='SS C')
    plt.legend()
    plt.subplot(3,1,3)
    plt.plot(t, L_arr, label='L')
    plt.axhline(ss['L_star'], color='r', ls='--', label='SS L')
    plt.legend()
    plt.xlabel('t')
    plt.tight_layout()
    plt.show()

    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.plot(L_arr, C_arr, K_arr)
    ax.scatter(L_arr[0], C_arr[0], K_arr[0], c='g', s=50, label='start')
    ax.scatter(L_arr[-1], C_arr[-1], K_arr[-1], c='r', s=50, label='end')
    ax.scatter(ss['L_star'], ss['C_star'], ss['K_star'], c='black', marker='*', s=100, label='SS')
    ax.set_xlabel('L'); ax.set_ylabel('C'); ax.set_zlabel('K')
    ax.legend()
    plt.show()

 ## Графики

In [110]:
def plot_3d_plotly(path, ss, T, K0):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter3d(
            x=path["K"],
            y=path["L"],          # <-- теперь L на оси Y (если хотите K, L, C)
            z=path["C"],          # <-- C на оси Z
            mode="lines+markers",
            name="Траектория RBC",
            line=dict(width=6, color=path["Y"], colorscale="Viridis"),
            marker=dict(size=4, color=np.arange(len(path["K"])), colorscale="Plasma"),
            hovertemplate="K=%{x:.4f}<br>L=%{y:.4f}<br>C=%{z:.4f}<extra></extra>",
        )
    )

    fig.add_trace(
        go.Scatter3d(
            x=[ss["K_star"]],
            y=[ss["L_star"]],
            z=[ss["C_star"]],
            mode="markers+text",
            name="Steady state",
            text=["SS"],
            textposition="top center",
            marker=dict(size=8, color="red", symbol="diamond"),
            hovertemplate="K*=%{x:.4f}<br>L*=%{y:.4f}<br>C*=%{z:.4f}<extra></extra>",
        )
    )

    title = f"3D-траектория модели RBC: (K<sub>t</sub>, L<sub>t</sub>, C<sub>t</sub>), K<sub>0</sub> = {K0:.4f}, T = {T}"

    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=20)),
        scene=dict(
            xaxis=dict(title="Капитал, K<sub>t</sub>", title_font=dict(size=16),
                       showbackground=True, tickfont=dict(size=11),
                       backgroundcolor="rgba(245,245,245,1)", gridcolor="lightgray"),
            yaxis=dict(title="Труд, L<sub>t</sub>", title_font=dict(size=16),
                       tickfont=dict(size=11), showbackground=True,
                       backgroundcolor="rgba(245,245,245,1)", gridcolor="lightgray"),
            zaxis=dict(title="Потребление, C<sub>t</sub>", title_font=dict(size=16),
                       tickfont=dict(size=11), showbackground=True,
                       backgroundcolor="rgba(245,245,245,1)", gridcolor="lightgray"),
            bgcolor="rgba(245,245,245,1)",
            aspectmode="manual",           # ручное управление масштабом осей
            aspectratio=dict(x=1, y=1, z=1), # одинаковый масштаб
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.5)   # начальный угол обзора
            )
        ),
        template="plotly_white",
        width=1000,
        height=700,
        margin=dict(l=0, r=0, b=0, t=80),
        legend=dict(x=0.02, y=0.98, font=dict(size=12)),
        font=dict(size=12),
    )

    config = {
        "toImageButtonOptions": {
            "format": "png",
            "filename": f"rbc_trajectory_3d_{K0:.4f}",
            "width": 1600,
            "height": 1200,
            "scale": 3,
        }
    }

    fig.show(config=config)
    return fig

In [94]:
def plot_time_series_grid_plotly(path, ss, T=None, K0=None):
    t = np.arange(len(path["K"]))

    mapping = [
        ("K", "<i>K</i><sub>t</sub>", "<i>K</i><sup>*</sup>"),
        ("C", "<i>C</i><sub>t</sub>", "<i>C</i><sup>*</sup>"),
        ("L", "<i>L</i><sub>t</sub>", "<i>L</i><sup>*</sup>"),
        ("Y", "<i>Y</i><sub>t</sub>", "<i>Y</i><sup>*</sup>"),
    ]

    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=[label for _, label, _ in mapping],
    )

    positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

    for (key, label, label_star), (r, c) in zip(mapping, positions):
        fig.add_trace(
            go.Scatter(
                x=t,
                y=path[key],
                mode="lines",
                name=label,
                showlegend=True,
            ),
            row=r,
            col=c,
        )

        fig.add_trace(
            go.Scatter(
                x=t,
                y=np.full_like(t, ss[f"{key}_star"], dtype=float),
                mode="lines",
                name=label_star,
                line=dict(dash="dash"),
                showlegend=True,
            ),
            row=r,
            col=c,
        )

        fig.update_xaxes(
            title_text="t",
            title_font=dict(size=16),
            tickfont=dict(size=12),
            row=r,
            col=c,
        )
        fig.update_yaxes(
            title_text=label,
            title_font=dict(size=16),
            tickfont=dict(size=12),
            row=r,
            col=c,
        )

    if K0 is not None and T is not None:
        title = f"Переходная динамика модели RBC, K<sub>0</sub> = {K0:.6f}, T = {T}"
    elif K0 is not None:
        title = f"Переходная динамика модели RBC, K<sub>0</sub> = {K0:.6f}"
    else:
        title = "Переходная динамика модели RBC"

    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=20)),
        height=800,
        width=1100,
        template="plotly_white",
        font=dict(size=12),
        legend=dict(
            x=1.02,
            y=1.0,
            xanchor="left",
            yanchor="top",
            font=dict(size=12),
        ),
        margin=dict(l=80, r=180, t=90, b=70),
    )

    # размер заголовков отдельных subplot
    for ann in fig.layout.annotations:
        ann.font = dict(size=16)

    config = {
        "toImageButtonOptions": {
            "format": "png",
            "filename": f"rbc_trajectory_2d_K0_{K0:.4f}_T_{T}",
            "width": 1600,
            "height": 1100,
            "scale": 3,
        }
    }

    fig.show(config=config)
    return fig

## Запуск

In [111]:
# p = Params()
# ss = steady_state(p)
# K0 = ss['K_star'] * 0.4   # начальный капитал ниже SS

# C0_opt, K_arr, C_arr, L_arr = find_C0_by_shooting(K0, p)
# if not np.isnan(C0_opt):
#     print(f"Оптимальное C0 = {C0_opt:.6f}")
#     plot_trajectories(K_arr, C_arr, L_arr, p)
#     # path =
#     # plot_3d_plotly(path, ss, T, K0):
# else:
#     print("Не удалось найти траекторию.")


p = Params()
ss = steady_state(p)
K0 = ss['K_star'] * 0.4   # начальный капитал ниже SS

C0_opt, path, T = find_C0_by_shooting(K0, p)
if path is not None:
    # Используем ваши функции отрисовки
    plot_time_series_grid_plotly(path, ss, T, K0)
    plot_3d_plotly(path, ss, T, K0)
else:
    print("Не удалось найти траекторию.")

Найдено C0 = 0.793123 за 31 итераций
